### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "xgb_optimized" / "XGBoost_thres0.5_2026-05-11" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.09,,,,,
1,0,cf_1,,,,,,,18.8688,,,0.0062,1,False,0.0743,0.0754
2,0,cf_2,,,,7,,,18.9745,,,0.1256,2,False,0.0743,0.0972
3,0,cf_3,2,,,,,,18.5952,,,0.1457,2,False,0.0743,0.1116
4,0,cf_4,,,,,,,17.5549,5,,0.165,2,True,0.0743,0.0099
5,0,cf_5,,,,6,,,17.4866,,,0.1627,2,False,0.0743,0.1366
6,0,cf_6,,,,6,,,17.0861,,,0.1839,2,False,0.0743,0.1366
7,0,cf_7,3,,,,,,16.7475,,,0.1809,2,False,0.0743,0.1413
8,0,cf_8,,,,7,,,16.6236,,,0.25,2,False,0.0743,0.1364
9,0,cf_9,,,,7,,,18.9745,7,,0.2506,3,True,0.0743,0.0051


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.09,,,,,
9,0,cf_4,,,,,,,17.5549,5,,0.165,2,True,0.0743,0.0099
10,0,cf_9,,,,7,,,18.9745,7,,0.2506,3,True,0.0743,0.0051
1,1,xin,3,4,1,2,3,0,22.3757,0,1.15,,,,,
11,1,cf_6,2,,,,1,,22.3757,,,0.2546,3,True,0.0484,0.0105
12,1,cf_8,,2,2,,,,22.3757,4,,0.2844,4,True,0.0484,0.0052
13,1,cf_10,,1,,,1,,22.3757,5,,0.3439,4,True,0.0484,0.0061
2,2,xin,5,3,1,1,4,0,22.694,7,1.13,,,,,
14,2,cf_2,3,,,,3,,22.68,,,0.1044,3,True,0.0647,0.0084
15,2,cf_4,,,,3,2,,22.6757,,,0.1254,3,True,0.0647,0.0185


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.09,,,,,
9,0,cf_9,,,,7,,,18.9745,7,,0.2506,3,True,0.0743,0.0051
1,1,xin,3,4,1,2,3,0,22.3757,0,1.15,,,,,
10,1,cf_6,2,,,,1,,22.3757,,,0.2546,3,True,0.0484,0.0105
2,2,xin,5,3,1,1,4,0,22.694,7,1.13,,,,,
11,2,cf_4,,,,3,2,,22.6757,,,0.1254,3,True,0.0647,0.0185
3,3,xin,3,3,6,6,2,0,24.3418,1,1.23,,,,,
12,3,cf_1,,,,,,,24.3375,,,0.0006,1,False,0.0048,0.0048
4,4,xin,5,4,2,7,1,0,21.2585,3,1.26,,,,,
13,4,cf_7,3,,,,,,21.2585,,,0.0625,2,True,0.0259,0.0118


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_4 ---
Predicted risk: 0.0099
Valid: True
Changes:
  bmi: 18.9866 → 17.5549
  dosprt: 0 → 5

--- cf_9 ---
Predicted risk: 0.0051
Valid: True
Changes:
  alcfreq: 4 → 7
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 7


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_6 ---
Predicted risk: 0.0105
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1

--- cf_8 ---
Predicted risk: 0.0052
Valid: True
Changes:
  eatveg: 4 → 2
  cgtsmok: 1 → 2
  dosprt: 0 → 4

--- cf_10 ---
Predicted risk: 0.0061
Valid: True
Changes:
  eatveg: 4 → 1
  slprl: 3 → 1
  dosprt: 0 → 5


=== Query index '2' ===
Task / Target